In [74]:
import os
import json
import networkx as nx

In [75]:
# Path to directory with all node logs
tlog_dir = "./logs-backup-20250802-175807"  # adjust as needed
# Prover information (node_id and log file)
prover_node_id = "12D3KooWDBcNby5ayhLeKTcAdqP492tmV9fJn59H67hKuHAzNYcV"   # fill the prover's node ID
prover_log_file = "node-2-log.log"  # fill the prover's log filename

### 2. Module: Map Node IDs to Log Filenames
Scan each log for the first "MDAG instance created" entry to build a mapping from node_id to filename.

In [76]:

def map_node_ids(log_dir):
    node_map = {}
    for fname in os.listdir(log_dir):
        fpath = os.path.join(log_dir, fname)
        with open(fpath, 'r') as f:
            for line in f:
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    continue
                if rec.get("msg") == "MDAG instance created":
                    node_map[rec["node_id"]] = fname
                    break
    return node_map

### 3. Module: Parse Neighbor Lists
Extract each node's neighbors from the "Initialized neighbors" log entry.

In [77]:
def parse_neighbors(log_dir, node_map):
    neighbors = {}
    for node_id, fname in node_map.items():
        fpath = os.path.join(log_dir, fname)
        with open(fpath, 'r') as f:
            for line in f:
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    continue
                if rec.get("msg") == "Initialized neighbors":
                    neighbors[node_id] = rec.get("neighbors", [])
                    break
    return neighbors

### 4. Module: Parse Message Events
Collect events for each node: initial send, valid processing, and validation failures, keyed by round and label.

In [78]:
def parse_message_events(log_dir, node_map):
    events = {nid: {'initial': [], 'valid': [], 'failed': []} for nid in node_map}
    for node_id, fname in node_map.items():
        fpath = os.path.join(log_dir, fname)
        with open(fpath, 'r') as f:
            for line in f:
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    continue
                msg = rec.get("msg", "")
                if msg == "Node is a prover, sending initial message":
                    events[node_id]['initial'].append(rec)
                elif msg == "Processing valid message":
                    events[node_id]['valid'].append(rec)
                elif msg == "Message validation failed":
                    events[node_id]['failed'].append(rec)
    return events

### 5. Module: Build & Analyze Propagation Graph
Using the neighbor topology and event records, track how the initial message propagates.

In [79]:
def analyze_propagation(node_map, neighbors, events, start_node):
    G = nx.DiGraph()
    G.add_node(start_node)
    # Seed with prover's initial labels
    initial_labels = [e.get('label') for e in events[start_node]['initial']]
    frontier = [(start_node, lbl) for lbl in initial_labels]

    visited = set()
    while frontier:
        current, label = frontier.pop(0)
        for nbr in neighbors.get(current, []):
            # Check if neighbor processed this label
            valid = any(ev.get('label') == label for ev in events.get(nbr, {}).get('valid', []))
            failed = any(ev.get('label') == label for ev in events.get(nbr, {}).get('failed', []))
            if valid:
                G.add_edge(current, nbr, label=label, status='valid')
                if (nbr, label) not in visited:
                    visited.add((nbr, label))
                    frontier.append((nbr, label))
            elif failed:
                G.add_edge(current, nbr, label=label, status='failed')
            # else: no record -> message not received or dropped
    return G

### 6. Usage Example
Put it all together to identify where propagation stops for the given prover.

In [80]:

# Step 1: map node IDs
node_map = map_node_ids(tlog_dir)
print(node_map)
# Step 2: parse neighbors
neighbors = parse_neighbors(tlog_dir, node_map)
print(neighbors)
# Step 3: parse events
events = parse_message_events(tlog_dir, node_map)
print(events)
# Step 4: analyze propagation
propagation_graph = analyze_propagation(node_map, neighbors, events, prover_node_id)
print(propagation_graph)
# Report unreachable or failed edges
for u, v, data in propagation_graph.edges(data=True):
    if data['status'] == 'failed':
        print(f"Propagation to {v} failed at edge {u}->{v} for label {data['label']}")
unreachable = set(node_map) - set(propagation_graph.nodes)
if unreachable:
    print("Nodes never reached:", unreachable)

{'12D3KooWCZCRkbyRm6duSBtmqnTqYMRvewZE4QjnWVfhwKEzjnk8': 'node-18-log.log', '12D3KooWHHNdgej1wongwMKqGBmGoiyQv3xsDKb743XmFJ2FunBc': 'node-3-log.log', '12D3KooWLBGMAXPydUf5c4syVu2dWhgFnMozk4Bubm27MYWNMSyE': 'node-11-log.log', '12D3KooWDBcNby5ayhLeKTcAdqP492tmV9fJn59H67hKuHAzNYcV': 'node-2-log.log', '12D3KooWC3hnG88GhcWXggUfztwCw2Ff5JdXiPDtz6cvMGbqjwsF': 'node-10-log.log', '12D3KooWPH41G1FMnYgUS8bzpM3kjM2p4wkTTVgR6XC16CBF7TjC': 'node-19-log.log', '12D3KooWDTNs7X4FH2ubNq7cdUVaqfWp9KamTsB7EWU9N5jKzRib': 'node-12-log.log', '12D3KooWLVbfroHXCNxKbJyG2wUMnkKWj1ZGArD3Fz2n8FesYu1A': 'node-9-log.log', '12D3KooWLmVZPbdT4iP9NDY2qxP1AJfqXbJPGGDcc31Bj5A3JPJS': 'node-8-log.log', '12D3KooWKv2mLKZVk7HMdqywwpQbHuPsr9LcWup7uiN9owVE28TK': 'node-1-log.log', '12D3KooWKpjwbN2iU9ujHNXSfkjJTnQ2gYuf9ZLRQaWJqGzZEYPo': 'node-13-log.log', '12D3KooWPVXns4vBvNuD6kna1CS8s9MnUDY82yTnupTnqxCgceoa': 'node-16-log.log', '12D3KooWPTKm8FFEJ2gxU5JwwvJZRCiFiNdYa79DtpJhZj4LTQh9': 'node-4-log.log', '12D3KooWFMcSrvNBrWHqw4qtjoCnR

In [81]:
n = "12D3KooWDBcNby5ayhLeKTcAdqP492tmV9fJn59H67hKuHAzNYcV"

In [82]:

for nbr in neighbors[n]:
    print(f"ngb: {nbr} - file {node_map[nbr]}")


ngb: 12D3KooWLVbfroHXCNxKbJyG2wUMnkKWj1ZGArD3Fz2n8FesYu1A - file node-9-log.log
ngb: 12D3KooWKv2mLKZVk7HMdqywwpQbHuPsr9LcWup7uiN9owVE28TK - file node-1-log.log
ngb: 12D3KooWKbf7BRVLnXNpHvFtv5vkPDPVVSCpNL9KZpkNy7tGkTfM - file node-17-log.log
ngb: 12D3KooWHHNdgej1wongwMKqGBmGoiyQv3xsDKb743XmFJ2FunBc - file node-3-log.log
